# Human grading: 30 answers

**Your job:** for each answer, decide **acceptable: yes / no**. Would a business user who asked this question be
correctly served by this answer?

**For each item you'll see:**
1. the question
2. **the system's answer**: its SQL, re-run live on the warehouse, and its written answer (if it has one)
3. **the right table(s)** for this question, with their description cards
4. **the reference query** (the frozen gold SQL) and its result
5. an empty cell for your own checks: `run("SELECT ...")`

**Which version wrote each answer is hidden.** Don't open `results/human_grading_key.json` until you're done.

**Tips for deciding**
- The numbers should match the reference result. Small rounding differences are fine.
- Extra columns or a ranked list with the right answer on top: your call. Would a business user be served?
- Watch the traps: valid vs all traffic, today/live is incomplete, reach = unique people (not impressions),
  weekly numbers should mention `days_in_week` when a week is incomplete.
- The rules are in `warehouse/rules.md`. "Yesterday" = 2017-05-12, "now" = 2017-05-13 23:59.

At the end, type your grades into the `my_grades` cell and run the last two cells. They save your grades into
`results/human_grading.csv` and compute the agreement (Cohen's kappa) with the AI grader.

In [1]:
import csv, json, sys
from pathlib import Path
import duckdb, pandas as pd, yaml
from IPython.display import display, Markdown

PROJECT = Path("..").resolve()
sys.path.append(str(PROJECT / "mcp_server"))
from safety import run_sql

con = duckdb.connect(str(PROJECT / "data" / "ads.duckdb"), read_only=True)
SHEET = PROJECT / "results" / "human_grading.csv"
items = {int(r["item"]): r for r in csv.DictReader(open(SHEET))}
questions = {q["question"]: q for q in map(json.loads, open(PROJECT / "benchmark" / "questions.jsonl"))}
pd.set_option("display.max_colwidth", 80)


def run(sql, limit=30):
    """Run any read-only SELECT yourself and see the result."""
    rows, columns, error = run_sql(con, sql)
    if error:
        print("ERROR:", error)
    else:
        display(pd.DataFrame(rows[:limit], columns=columns))


def show(k):
    item = items[k]
    q = questions[item["question"]]
    display(Markdown(f"### Item {k} ({q['tier']})\n**Question:** {item['question']}"))

    display(Markdown("**1. The system's SQL** (re-run now):"))
    if item["sql"].strip():
        print(item["sql"])
        run(item["sql"])
    else:
        print("(no SQL: the system said 'not sure')")
    if item["written_answer"].strip():
        display(Markdown("**The system's written answer:**"))
        print(item["written_answer"])

    display(Markdown("**2. The right table(s) for this question:**"))
    for table in q["gold_tables"]:
        card = yaml.safe_load(open(PROJECT / "catalog" / "cards" / f"{table}.yaml"))
        print(f"- {table}: {' '.join(card['description'].split())}")
        print(f"    grain: {card['grain']} | freshness: {card['freshness']} | traffic: {card['traffic']}")
    if q["trap"]:
        print(f"Known trap: {q['trap']}")

    display(Markdown("**3. The reference (gold) query and its result:**"))
    gold_sql = (PROJECT / "benchmark" / q["gold_sql"]).read_text().strip()
    print(gold_sql)
    run(gold_sql)

In [2]:
show(1)

### Item 1 (medium)
**Question:** Reach among men for campaign 226431 on 8 May 2017?

**1. The system's SQL** (re-run now):

select count(distinct f.user_id) as reach_men
from fact_ad_events f
join dim_user u on f.user_id = u.user_id
where f.event_date = date '2017-05-08'
  and f.campaign_id = 226431
  and u.gender = 'male';


,reach_men
0,23


**The system's written answer:**

Reach (unique men who saw campaign 226431) on 2017-05-08 = 23 men.

Caveats: data is final for 2017-05-08 (the dataset covers 2017-05-06 to 2017-05-13). I counted distinct user_id in fact_ad_events joined to dim_user gender = 'male'.

SQL used:
```sql
select count(distinct f.user_id) as reach_men
from fact_ad_events f
join dim_user u on f.user_id = u.user_id
where f.event_date = date '2017-05-08'
  and f.campaign_id = 226431
  and u.gender = 'male';
```


**2. The right table(s) for this question:**

- fact_ad_events: Every single ad impression (one ad shown to one user) with exact time, click flag, ad, campaign, brand, category, placement and price. Very large; use summary tables when they fit the question.
    grain: one row per impression | freshness: includes_today | traffic: all
- dim_user: User profile for each user - gender, age level, spending level, shopping depth, student flag, city level. Use it to split results by audience. Some users have no profile.
    grain: one row per user | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT COUNT(DISTINCT e.user_id) AS unique_users FROM fact_ad_events e JOIN dim_user u ON u.user_id = e.user_id WHERE e.campaign_id = 226431 AND u.gender = 'male' AND e.event_date = '2017-05-08'


,unique_users
0,23


In [3]:
# Item 1: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [4]:
show(2)

### Item 2 (tricky)
**Question:** Campaign 141116: clicks at 23:00-23:59 on 8 May 2017 after removing bot traffic?

**1. The system's SQL** (re-run now):

SELECT clicks
FROM agg_campaign_hourly_valid
WHERE campaign_id = 141116
  AND event_hour = TIMESTAMP '2017-05-08 23:00:00'


,clicks
0,3


**2. The right table(s) for this question:**

- agg_campaign_hourly_valid: Hourly impressions, clicks, CTR and estimated value per campaign with suspicious / invalid / bot traffic removed. Includes today.
    grain: one row per campaign per hour | freshness: includes_today | traffic: valid
Known trap: all-traffic hourly table; daily valid table


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_campaign_hourly_valid WHERE campaign_id = 141116 AND event_hour = '2017-05-08 23:00:00'


,clicks
0,3


In [5]:
# Item 2: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [6]:
show(3)

### Item 3 (tricky)
**Question:** How many unique people saw brand 406642's ads yesterday?

**1. The system's SQL** (re-run now):

SELECT unique_users
FROM reach_brand_daily
WHERE brand_id = 406642
  AND event_date = DATE '2017-05-12'


,unique_users
0,235


**2. The right table(s) for this question:**

- reach_brand_daily: Reach per brand per day - number of unique users (people) who saw the brand's ads and unique users who clicked.
    grain: one row per brand per day | freshness: final | traffic: all
Known trap: impressions (agg_brand_daily) instead of unique people


**3. The reference (gold) query and its result:**

SELECT unique_users FROM reach_brand_daily WHERE brand_id = 406642 AND event_date = '2017-05-12'


,unique_users
0,235


In [7]:
# Item 3: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [8]:
show(4)

### Item 4 (easy)
**Question:** Number of clicks for ad 650916 on 11 May 2017?

**1. The system's SQL** (re-run now):

SELECT clicks
FROM agg_ad_daily
WHERE ad_id = 650916
  AND event_date = DATE '2017-05-11'


,clicks
0,12


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_ad_daily WHERE ad_id = 650916 AND event_date = '2017-05-11'


,clicks
0,12


In [9]:
# Item 4: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [10]:
show(5)

### Item 5 (tricky)
**Question:** How many unique people saw brand 406642's ads yesterday?

**1. The system's SQL** (re-run now):

SELECT event_date, brand_id, unique_users
FROM reach_brand_daily
WHERE event_date = DATE '2017-05-12'
  AND brand_id = 406642;


,event_date,brand_id,unique_users
0,2017-05-12,406642,235


**The system's written answer:**

235 unique people saw brand 406642's ads yesterday (2017-05-12).

SQL used:
```sql
SELECT event_date, brand_id, unique_users
FROM reach_brand_daily
WHERE event_date = DATE '2017-05-12'
  AND brand_id = 406642;
```


**2. The right table(s) for this question:**

- reach_brand_daily: Reach per brand per day - number of unique users (people) who saw the brand's ads and unique users who clicked.
    grain: one row per brand per day | freshness: final | traffic: all
Known trap: impressions (agg_brand_daily) instead of unique people


**3. The reference (gold) query and its result:**

SELECT unique_users FROM reach_brand_daily WHERE brand_id = 406642 AND event_date = '2017-05-12'


,unique_users
0,235


In [11]:
# Item 5: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [12]:
show(6)

### Item 6 (easy)
**Question:** How many ads does campaign 116859 have?

**1. The system's SQL** (re-run now):

SELECT num_ads
FROM dim_campaign
WHERE campaign_id = 116859


,num_ads
0,21


**2. The right table(s) for this question:**

- dim_campaign: Lookup of every campaign with its advertiser and how many ads it has.
    grain: one row per campaign | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT num_ads FROM dim_campaign WHERE campaign_id = 116859


,num_ads
0,21


In [13]:
# Item 6: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [14]:
show(7)

### Item 7 (tricky)
**Question:** Campaign 141116: clicks at 23:00-23:59 on 8 May 2017 after removing bot traffic?

**1. The system's SQL** (re-run now):

SELECT event_hour, campaign_id, impressions, clicks, ctr
FROM agg_campaign_hourly_valid
WHERE campaign_id = 141116
  AND event_hour = TIMESTAMP '2017-05-08 23:00:00';


,event_hour,campaign_id,impressions,clicks,ctr
0,2017-05-08 23:00:00,141116,56,3,0.053571


**The system's written answer:**

Campaign 141116 had 3 clicks (56 impressions, CTR = 3/56 = 0.05357142857142857) during 23:00–23:59 on 2017-05-08 after bot/suspicious traffic was removed.

SQL:
```sql
SELECT event_hour, campaign_id, impressions, clicks, ctr
FROM agg_campaign_hourly_valid
WHERE campaign_id = 141116
  AND event_hour = TIMESTAMP '2017-05-08 23:00:00';
```



**2. The right table(s) for this question:**

- agg_campaign_hourly_valid: Hourly impressions, clicks, CTR and estimated value per campaign with suspicious / invalid / bot traffic removed. Includes today.
    grain: one row per campaign per hour | freshness: includes_today | traffic: valid
Known trap: all-traffic hourly table; daily valid table


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_campaign_hourly_valid WHERE campaign_id = 141116 AND event_hour = '2017-05-08 23:00:00'


,clicks
0,3


In [15]:
# Item 7: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [16]:
show(8)

### Item 8 (easy)
**Question:** Number of campaigns for advertiser 125660?

**1. The system's SQL** (re-run now):

SELECT advertiser_id, num_campaigns, num_ads
FROM dim_advertiser
WHERE advertiser_id = 125660;


,advertiser_id,num_campaigns,num_ads
0,125660,9,10


**The system's written answer:**

Advertiser 125660 has 9 campaigns (and 10 ads).

SQL used:
```sql
SELECT advertiser_id, num_campaigns, num_ads
FROM dim_advertiser
WHERE advertiser_id = 125660;
```


**2. The right table(s) for this question:**

- dim_advertiser: Lookup of every advertiser (customer) with how many campaigns and ads it runs.
    grain: one row per advertiser | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT num_campaigns FROM dim_advertiser WHERE advertiser_id = 125660


,num_campaigns
0,9


In [17]:
# Item 8: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [18]:
show(9)

### Item 9 (easy)
**Question:** How much does the product in ad 748949 cost?

**1. The system's SQL** (re-run now):

SELECT price
FROM dim_ad
WHERE ad_id = 748949


,price
0,339.0


**2. The right table(s) for this question:**

- dim_ad: Lookup of every ad with its campaign, advertiser, brand, product category and item price. Use it to find which campaign, brand or advertiser an ad belongs to.
    grain: one row per ad | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT price FROM dim_ad WHERE ad_id = 748949


,price
0,339.0


In [19]:
# Item 9: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [20]:
show(10)

### Item 10 (easy)
**Question:** What was the CTR of campaign 112525 on 7 May 2017?

**1. The system's SQL** (re-run now):

SELECT event_date, campaign_id, impressions, clicks, ctr, CASE WHEN impressions=0 THEN NULL ELSE clicks::DOUBLE / impressions END AS ctr_calc
FROM agg_campaign_daily
WHERE campaign_id = 112525
  AND event_date = DATE '2017-05-07';


,event_date,campaign_id,impressions,clicks,ctr,ctr_calc
0,2017-05-07,112525,161,10,0.062112,0.062112


**The system's written answer:**

Campaign 112525 on 2017-05-07: impressions = 161, clicks = 10, CTR = 0.062111801242236024.

Caveats: this is the final daily table (all traffic, up to yesterday). If you want bot-filtered CTR use agg_campaign_daily_valid instead.

SQL used:
```sql
SELECT event_date, campaign_id, impressions, clicks, ctr, CASE WHEN impressions=0 THEN NULL ELSE clicks::DOUBLE / impressions END AS ctr_calc
FROM agg_campaign_daily
WHERE campaign_id = 112525
  AND event_date = DATE '2017-05-07';
```


**2. The right table(s) for this question:**

- agg_campaign_daily: Final daily impressions, clicks, CTR and estimated value per campaign, all traffic, up to yesterday.
    grain: one row per campaign per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT ctr FROM agg_campaign_daily WHERE campaign_id = 112525 AND event_date = '2017-05-07'


,ctr
0,0.062112


In [21]:
# Item 10: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [22]:
show(11)

### Item 11 (medium)
**Question:** Yesterday's clicks for advertiser 179351, only ads in category 8973?

**1. The system's SQL** (re-run now):

SELECT
  a.event_date,
  d.advertiser_id,
  d.category_id,
  SUM(a.clicks) AS clicks
FROM agg_ad_daily a
JOIN dim_ad d USING (ad_id)
WHERE a.event_date = DATE '2017-05-12'
  AND d.advertiser_id = 179351
  AND d.category_id = 8973
GROUP BY a.event_date, d.advertiser_id, d.category_id


,event_date,advertiser_id,category_id,clicks
0,2017-05-12,179351,8973,3


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all
- dim_ad: Lookup of every ad with its campaign, advertiser, brand, product category and item price. Use it to find which campaign, brand or advertiser an ad belongs to.
    grain: one row per ad | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(d.clicks) AS clicks FROM agg_ad_daily d JOIN dim_ad a ON a.ad_id = d.ad_id WHERE a.advertiser_id = 179351 AND a.category_id = 8973 AND d.event_date = '2017-05-12'


,clicks
0,3


In [23]:
# Item 11: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [24]:
show(12)

### Item 12 (easy)
**Question:** Top category by clicks on 11 May 2017?

**1. The system's SQL** (re-run now):

SELECT category_id, clicks
FROM agg_category_daily
WHERE event_date = DATE '2017-05-11'
ORDER BY clicks DESC
LIMIT 1


,category_id,clicks
0,6261,23099


**2. The right table(s) for this question:**

- agg_category_daily: Final daily impressions, clicks, CTR and estimated value per product category.
    grain: one row per category per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT category_id, clicks FROM agg_category_daily WHERE event_date = '2017-05-11' ORDER BY clicks DESC LIMIT 1


,category_id,clicks
0,6261,23099


In [25]:
# Item 12: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [26]:
show(13)

### Item 13 (medium)
**Question:** How many clicks did brand 166215 get from high-spending users (spending level 3) from 6 to 12 May?

**1. The system's SQL** (re-run now):

SELECT SUM(f.is_click) AS clicks
FROM fact_ad_events f
JOIN dim_user u ON f.user_id = u.user_id
WHERE f.brand_id = 166215
  AND u.spending_level = 3
  AND f.event_date BETWEEN DATE '2017-05-06' AND DATE '2017-05-12';


,clicks
0,4


**The system's written answer:**

Brand 166215 received 4 clicks from high-spending users (spending_level = 3) between 2017-05-06 and 2017-05-12 inclusive.

SQL used:
```sql
SELECT SUM(f.is_click) AS clicks
FROM fact_ad_events f
JOIN dim_user u ON f.user_id = u.user_id
WHERE f.brand_id = 166215
  AND u.spending_level = 3
  AND f.event_date BETWEEN DATE '2017-05-06' AND DATE '2017-05-12';
```

Caveats: data covers 2017-05-06 to 2017-05-13 and includes today; I used May 6–12 as requested (yesterday is May 12). Users without a profile (no dim_user row) were excluded.


... (2 outputs left out)


    grain: one row per impression | freshness: includes_today | traffic: all
- dim_user: User profile for each user - gender, age level, spending level, shopping depth, student flag, city level. Use it to split results by audience. Some users have no profile.
    grain: one row per user | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(e.is_click) AS clicks FROM fact_ad_events e JOIN dim_user u ON u.user_id = e.user_id WHERE e.brand_id = 166215 AND u.spending_level = 3 AND e.event_date <= '2017-05-12'


,clicks
0,4


In [27]:
# Item 13: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [28]:
show(14)

### Item 14 (tricky)
**Question:** How many clicks has campaign 390070 had today so far?

**1. The system's SQL** (re-run now):

SELECT
  (
    COALESCE(
      (SELECT SUM(clicks) FROM agg_campaign_hourly
       WHERE campaign_id = 390070
         AND event_hour >= TIMESTAMP '2017-05-13 00:00:00'
         AND event_hour <  TIMESTAMP '2017-05-13 18:00:00'
      ), 0
    )
    +
    COALESCE(
      (SELECT clicks FROM live_campaign_last6h WHERE campaign_id = 390070), 0
    )
  ) AS clicks_today


,clicks_today
0,11


**2. The right table(s) for this question:**

- agg_campaign_hourly: Impressions, clicks, CTR and estimated value per campaign per hour, all traffic. Includes today, so today's hours are incomplete.


    grain: one row per campaign per hour | freshness: includes_today | traffic: all
Known trap: daily tables stop at yesterday; live tables only cover 6 hours


**3. The reference (gold) query and its result:**

SELECT SUM(clicks) AS clicks FROM agg_campaign_hourly WHERE campaign_id = 390070 AND event_hour >= '2017-05-13'


,clicks
0,11


In [29]:
# Item 14: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [30]:
show(15)

### Item 15 (tricky)
**Question:** Campaign 58231: clicks at 18:00-18:59 on 10 May 2017 after removing bot traffic?

**1. The system's SQL** (re-run now):

SELECT clicks
FROM agg_campaign_hourly_valid
WHERE campaign_id = 58231
  AND event_hour = '2017-05-10 18:00:00'


,clicks
0,13


**2. The right table(s) for this question:**

- agg_campaign_hourly_valid: Hourly impressions, clicks, CTR and estimated value per campaign with suspicious / invalid / bot traffic removed. Includes today.
    grain: one row per campaign per hour | freshness: includes_today | traffic: valid
Known trap: all-traffic hourly table; daily valid table


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_campaign_hourly_valid WHERE campaign_id = 58231 AND event_hour = '2017-05-10 18:00:00'


,clicks
0,13


In [31]:
# Item 15: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [32]:
show(16)

### Item 16 (medium)
**Question:** Yesterday's clicks for advertiser 56581, only ads in category 6957?

**1. The system's SQL** (re-run now):

SELECT SUM(is_click) AS clicks
FROM fact_ad_events
WHERE event_date = DATE '2017-05-12'
  AND advertiser_id = 56581
  AND category_id = 6957


,clicks
0,4


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


- dim_ad: Lookup of every ad with its campaign, advertiser, brand, product category and item price. Use it to find which campaign, brand or advertiser an ad belongs to.
    grain: one row per ad | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(d.clicks) AS clicks FROM agg_ad_daily d JOIN dim_ad a ON a.ad_id = d.ad_id WHERE a.advertiser_id = 56581 AND a.category_id = 6957 AND d.event_date = '2017-05-12'


,clicks
0,4


In [33]:
# Item 16: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [34]:
show(17)

### Item 17 (easy)
**Question:** Number of clicks for ad 650916 on 11 May 2017?

**1. The system's SQL** (re-run now):

SELECT clicks
FROM agg_ad_daily
WHERE ad_id = 650916
  AND event_date = DATE '2017-05-11'


,clicks
0,12


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_ad_daily WHERE ad_id = 650916 AND event_date = '2017-05-11'


,clicks
0,12


In [35]:
# Item 17: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [36]:
show(18)

### Item 18 (easy)
**Question:** Estimated value (clicks x price) for advertiser 155304 on 8 May 2017?

**1. The system's SQL** (re-run now):

SELECT event_date, advertiser_id, clicks, est_value
FROM agg_advertiser_daily
WHERE advertiser_id = 155304
  AND event_date = DATE '2017-05-08';


,event_date,advertiser_id,clicks,est_value
0,2017-05-08,155304,159,20902.0


**The system's written answer:**

Advertiser 155304 on 2017-05-08: clicks = 159 and estimated value (clicks × price) = 20902.0.

Caveats: agg_advertiser_daily is final data through 2017-05-12; 2017-05-08 is within the final range.

SQL used:
```sql
SELECT event_date, advertiser_id, clicks, est_value
FROM agg_advertiser_daily
WHERE advertiser_id = 155304
  AND event_date = DATE '2017-05-08';
```


**2. The right table(s) for this question:**

- agg_advertiser_daily: Final daily impressions, clicks, CTR and estimated value per advertiser (all its campaigns together).
    grain: one row per advertiser per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT est_value FROM agg_advertiser_daily WHERE advertiser_id = 155304 AND event_date = '2017-05-08'


,est_value
0,20902.0


In [37]:
# Item 18: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [38]:
show(19)

### Item 19 (tricky)
**Question:** What was the CTR for campaign 101417 on 7 May 2017, split by gender?

**1. The system's SQL** (re-run now):

SELECT
  gender,
  ctr
FROM agg_campaign_daily_by_audience
WHERE campaign_id = 101417
  AND event_date = DATE '2017-05-07'
ORDER BY gender


,gender,ctr
0,female,0.078947
1,female,0.024390
2,female,0.000000
3,female,0.102564
4,female,0.000000
5,female,0.000000
6,male,0.000000
7,male,0.000000
8,male,0.000000
9,male,0.111111


**2. The right table(s) for this question:**

- agg_campaign_daily_by_audience: Final daily impressions, clicks and CTR per campaign split by user gender and age level (audience breakdown / demographics). Users without a profile are 'unknown'.
    grain: one row per campaign per day per gender per age level | freshness: final | traffic: all
Known trap: the audience table has several rows per gender (one per age level)


**3. The reference (gold) query and its result:**

SELECT gender, SUM(clicks) / SUM(impressions) AS ctr FROM agg_campaign_daily_by_audience WHERE campaign_id = 101417 AND event_date = '2017-05-07' GROUP BY gender ORDER BY gender


,gender,ctr
0,female,0.076923
1,male,0.096774
2,unknown,0.055556


In [39]:
# Item 19: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [40]:
show(20)

### Item 20 (easy)
**Question:** How many clicks did ad 779234 get on 10 May 2017?

**1. The system's SQL** (re-run now):

SELECT COALESCE(SUM(clicks), 0) AS clicks
FROM agg_ad_daily
WHERE event_date = DATE '2017-05-10'
  AND ad_id = 779234


,clicks
0,11


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_ad_daily WHERE ad_id = 779234 AND event_date = '2017-05-10'


,clicks
0,11


In [41]:
# Item 20: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [42]:
show(21)

### Item 21 (medium)
**Question:** How many clicks did advertiser 215611's ads in category 4385 get yesterday?

**1. The system's SQL** (re-run now):

SELECT
  SUM(is_click) AS clicks
FROM
  fact_ad_events
WHERE
  event_date = DATE '2017-05-12'
  AND advertiser_id = 215611
  AND category_id = 4385;


,clicks
0,3


**The system's written answer:**

Advertiser 215611's ads in category 4385 received 3 clicks yesterday (2017-05-12).

SQL used:
```sql
SELECT
  SUM(is_click) AS clicks
FROM
  fact_ad_events
WHERE
  event_date = DATE '2017-05-12'
  AND advertiser_id = 215611
  AND category_id = 4385;
```

Caveats: data comes from the raw impression-level table (fact_ad_events) which includes all traffic and is complete through 2017-05-12.


... (2 outputs left out)


    grain: one row per ad per day | freshness: final | traffic: all
- dim_ad: Lookup of every ad with its campaign, advertiser, brand, product category and item price. Use it to find which campaign, brand or advertiser an ad belongs to.
    grain: one row per ad | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(d.clicks) AS clicks FROM agg_ad_daily d JOIN dim_ad a ON a.ad_id = d.ad_id WHERE a.advertiser_id = 215611 AND a.category_id = 4385 AND d.event_date = '2017-05-12'


,clicks
0,3


In [43]:
# Item 21: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [44]:
show(22)

### Item 22 (easy)
**Question:** Number of clicks for ad 650916 on 11 May 2017?

**1. The system's SQL** (re-run now):

SELECT clicks
FROM agg_ad_daily
WHERE ad_id = 650916
  AND event_date = DATE '2017-05-11'


,clicks
0,12


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT clicks FROM agg_ad_daily WHERE ad_id = 650916 AND event_date = '2017-05-11'


,clicks
0,12


In [45]:
# Item 22: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [46]:
show(23)

### Item 23 (medium)
**Question:** How many clicks did brand 166215 get from high-spending users (spending level 3) from 6 to 12 May?

**1. The system's SQL** (re-run now):

SELECT SUM(fe.is_click) AS clicks
FROM fact_ad_events AS fe
JOIN dim_user AS u ON fe.user_id = u.user_id
WHERE fe.event_date BETWEEN DATE '2017-05-06' AND DATE '2017-05-12'
  AND fe.brand_id = 166215
  AND u.spending_level = 3


,clicks
0,4


**2. The right table(s) for this question:**

- fact_ad_events: Every single ad impression (one ad shown to one user) with exact time, click flag, ad, campaign, brand, category, placement and price. Very large; use summary tables when they fit the question.
    grain: one row per impression | freshness: includes_today | traffic: all
- dim_user: User profile for each user - gender, age level, spending level, shopping depth, student flag, city level. Use it to split results by audience. Some users have no profile.
    grain: one row per user | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(e.is_click) AS clicks FROM fact_ad_events e JOIN dim_user u ON u.user_id = e.user_id WHERE e.brand_id = 166215 AND u.spending_level = 3 AND e.event_date <= '2017-05-12'


,clicks
0,4


In [47]:
# Item 23: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [48]:
show(24)

### Item 24 (medium)
**Question:** For campaign 94399 yesterday, what CTR did heavy shoppers have?

**1. The system's SQL** (re-run now):

SELECT
  CASE WHEN impressions = 0 THEN 0.0 ELSE clicks::DOUBLE / impressions::DOUBLE END AS ctr
FROM (
  SELECT
    COUNT(*) AS impressions,
    SUM(is_click) AS clicks
  FROM fact_ad_events e
  JOIN dim_user u ON e.user_id = u.user_id
  WHERE e.event_date = DATE '2017-05-12'
    AND e.campaign_id = 94399
    AND u.shopping_depth = 5
) t


,ctr
0,0.0


**2. The right table(s) for this question:**

- fact_ad_events: Every single ad impression (one ad shown to one user) with exact time, click flag, ad, campaign, brand, category, placement and price. Very large; use summary tables when they fit the question.


    grain: one row per impression | freshness: includes_today | traffic: all
- dim_user: User profile for each user - gender, age level, spending level, shopping depth, student flag, city level. Use it to split results by audience. Some users have no profile.
    grain: one row per user | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(e.is_click) / COUNT(*) AS ctr FROM fact_ad_events e JOIN dim_user u ON u.user_id = e.user_id WHERE e.campaign_id = 94399 AND u.shopping_depth = 3 AND e.event_date = '2017-05-12'


,ctr
0,0.031339


In [49]:
# Item 24: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [50]:
show(25)

### Item 25 (tricky)
**Question:** What was campaign 143747's CTR yesterday, excluding suspicious clicks?

**1. The system's SQL** (re-run now):

SELECT event_date, campaign_id, impressions, clicks, ctr, CASE WHEN impressions=0 THEN NULL ELSE clicks::DOUBLE / impressions END AS ctr_calc
FROM agg_campaign_daily_valid
WHERE campaign_id = 143747
  AND event_date = DATE '2017-05-12';


,event_date,campaign_id,impressions,clicks,ctr,ctr_calc
0,2017-05-12,143747,142,8,0.056338,0.056338


**The system's written answer:**

Campaign 143747's CTR yesterday (2017-05-12), with suspicious clicks removed, was 0.056338028169014086 (clicks / impressions = 8 / 142).

SQL used:
```sql
SELECT event_date, campaign_id, impressions, clicks, ctr, CASE WHEN impressions=0 THEN NULL ELSE clicks::DOUBLE / impressions END AS ctr_calc
FROM agg_campaign_daily_valid
WHERE campaign_id = 143747
  AND event_date = DATE '2017-05-12';
```


... (2 outputs left out)


    grain: one row per campaign per day | freshness: final | traffic: valid
Known trap: all-traffic table agg_campaign_daily


**3. The reference (gold) query and its result:**

SELECT ctr FROM agg_campaign_daily_valid WHERE campaign_id = 143747 AND event_date = '2017-05-12'


,ctr
0,0.056338


In [51]:
# Item 25: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [52]:
show(26)

### Item 26 (tricky)
**Question:** Total reach (unique users) of campaign 207957 over the final period?

**1. The system's SQL** (re-run now):

SELECT campaign_id, unique_users
FROM reach_campaign_total
WHERE campaign_id = 207957;


,campaign_id,unique_users
0,207957,1411


**The system's written answer:**

Campaign 207957 reached 1,411 unique users over the final period (May 6–12, finalised data).

SQL used:
```sql
SELECT campaign_id, unique_users
FROM reach_campaign_total
WHERE campaign_id = 207957;
```


**2. The right table(s) for this question:**

- reach_campaign_total: Total reach per campaign over the whole final period (6-12 May) - unique users, unique clickers and average frequency (impressions per person).
    grain: one row per campaign | freshness: final | traffic: all
Known trap: adding up reach_campaign_daily counts people twice


**3. The reference (gold) query and its result:**

SELECT unique_users FROM reach_campaign_total WHERE campaign_id = 207957


,unique_users
0,1411


In [53]:
# Item 26: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [54]:
show(27)

### Item 27 (easy)
**Question:** Which advertiser runs campaign 234670?

**1. The system's SQL** (re-run now):

SELECT advertiser_id
FROM dim_campaign
WHERE campaign_id = 234670


,advertiser_id
0,159990


**2. The right table(s) for this question:**

- dim_campaign: Lookup of every campaign with its advertiser and how many ads it has.
    grain: one row per campaign | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT advertiser_id FROM dim_campaign WHERE campaign_id = 234670


,advertiser_id
0,159990


In [55]:
# Item 27: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [56]:
show(28)

### Item 28 (medium)
**Question:** How many clicks did advertiser 215611's ads in category 4385 get yesterday?

**1. The system's SQL** (re-run now):

SELECT COALESCE(SUM(a.clicks), 0) AS clicks
FROM agg_ad_daily a
JOIN dim_ad d ON a.ad_id = d.ad_id
WHERE a.event_date = DATE '2017-05-12'
  AND d.advertiser_id = 215611
  AND d.category_id = 4385


,clicks
0,3


**2. The right table(s) for this question:**

- agg_ad_daily: Final daily impressions, clicks, CTR and estimated value per ad, all traffic, up to yesterday.
    grain: one row per ad per day | freshness: final | traffic: all


- dim_ad: Lookup of every ad with its campaign, advertiser, brand, product category and item price. Use it to find which campaign, brand or advertiser an ad belongs to.
    grain: one row per ad | freshness: static | traffic: n/a


**3. The reference (gold) query and its result:**

SELECT SUM(d.clicks) AS clicks FROM agg_ad_daily d JOIN dim_ad a ON a.ad_id = d.ad_id WHERE a.advertiser_id = 215611 AND a.category_id = 4385 AND d.event_date = '2017-05-12'


,clicks
0,3


In [57]:
# Item 28: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [58]:
show(29)

### Item 29 (easy)
**Question:** Top category by clicks on 11 May 2017?

**1. The system's SQL** (re-run now):

SELECT category_id, clicks
FROM agg_category_daily
WHERE event_date = DATE '2017-05-11'
ORDER BY clicks DESC
LIMIT 1


,category_id,clicks
0,6261,23099


**2. The right table(s) for this question:**

- agg_category_daily: Final daily impressions, clicks, CTR and estimated value per product category.
    grain: one row per category per day | freshness: final | traffic: all


**3. The reference (gold) query and its result:**

SELECT category_id, clicks FROM agg_category_daily WHERE event_date = '2017-05-11' ORDER BY clicks DESC LIMIT 1


,category_id,clicks
0,6261,23099


In [59]:
# Item 29: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

In [60]:
show(30)

### Item 30 (tricky)
**Question:** What was campaign 143747's CTR yesterday, excluding suspicious clicks?

**1. The system's SQL** (re-run now):

SELECT ctr
FROM agg_campaign_daily_valid
WHERE event_date = DATE '2017-05-12'
  AND campaign_id = 143747


,ctr
0,0.056338


**2. The right table(s) for this question:**

- agg_campaign_daily_valid: Final daily impressions, clicks, CTR and estimated value per campaign with suspicious / invalid / bot traffic removed.
    grain: one row per campaign per day | freshness: final | traffic: valid
Known trap: all-traffic table agg_campaign_daily


**3. The reference (gold) query and its result:**

SELECT ctr FROM agg_campaign_daily_valid WHERE campaign_id = 143747 AND event_date = '2017-05-12'


,ctr
0,0.056338


In [61]:
# Item 30: your own checks, e.g.
# run("SELECT * FROM agg_campaign_daily WHERE campaign_id = 123 LIMIT 5")

## Your grades
Write `"yes"` or `"no"` for every item (and a short note if you like), then run the next two cells.

In [7]:
my_grades = {
    1: ("yes", ""),   # (yes/no, note)
    2: ("yes", ""),   # (yes/no, note)
    3: ("yes", ""),   # (yes/no, note)
    4: ("yes", ""),   # (yes/no, note)
    5: ("yes", ""),   # (yes/no, note)
    6: ("yes", ""),   # (yes/no, note)
    7: ("yes", ""),   # (yes/no, note)
    8: ("yes", ""),   # (yes/no, note)
    9: ("yes", ""),   # (yes/no, note)
    10: ("yes", ""),   # (yes/no, note)
    11: ("yes", ""),   # (yes/no, note)
    12: ("yes", ""),   # (yes/no, note)
    13: ("yes", ""),   # (yes/no, note)
    14: ("yes", ""),   # (yes/no, note)
    15: ("yes", ""),   # (yes/no, note)
    16: ("yes", ""),   # (yes/no, note)
    17: ("yes", ""),   # (yes/no, note)
    18: ("yes", ""),   # (yes/no, note)
    19: ("no", ""),   # (yes/no, note)
    20: ("yes", ""),   # (yes/no, note)
    21: ("yes", ""),   # (yes/no, note)
    22: ("yes", ""),   # (yes/no, note)
    23: ("yes", ""),   # (yes/no, note)
    24: ("no", ""),   # (yes/no, note)
    25: ("yes", ""),   # (yes/no, note)
    26: ("yes", ""),   # (yes/no, note)
    27: ("yes", ""),   # (yes/no, note)
    28: ("yes", ""),   # (yes/no, note)
    29: ("yes", ""),   # (yes/no, note)
    30: ("yes", ""),   # (yes/no, note)
}

In [8]:
# save your grades into the sheet
rows = list(csv.DictReader(open(SHEET)))
for r in rows:
    grade, note = my_grades[int(r["item"])]
    r["your_acceptable (yes/no)"], r["your_note"] = grade.strip().lower(), note
with open(SHEET, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
missing = [k for k, (g, _) in my_grades.items() if g.strip().lower() not in ("yes", "no")]
print(f"Saved. Graded {len(rows) - len(missing)}/{len(rows)}." + (f" Still missing: {missing}" if missing else ""))

Saved. Graded 30/30.


In [9]:
# agreement with the AI grader (Cohen's kappa; target >= 0.6)
sys.path.append(str(PROJECT / "harness"))
import os
os.chdir(PROJECT)
import judge
judge.kappa_with_human()

30 answers graded by you. Same verdict as the AI grader: 29/30. Cohen's kappa = 0.65 (target >= 0.6)
